# Exercise 1: Getting Started with PyPSA

**Presenter** <br>
Priyesh Gosai <br>
Energy Systems Modeller <br>
priyesh@innovateimpact.com <br>

**Course Details**<br>
Modelling Integrated Power Markets <br>
Johannesburg, South Africa <br>
20-24 April 2026 <br>



## Prepare Google Colab Environment

In [ ]:
lesson_number = 1
print(f'lesson{lesson_number}')

In [ ]:
#@title Connect to Google Drive {display-mode:"form"}
CONNECT_TO_DRIVE = True #@param {type:"boolean"}

import os

if CONNECT_TO_DRIVE:
    from google.colab import drive
    # Mount Google Drive
    drive.mount('/content/drive')

    # Define the desired working directory path
    working_dir = f'/content/drive/MyDrive/ich-modelling-2026'
    lesson_folder = f'lesson-{lesson_number}'
    lesson_dir = os.path.join(working_dir, lesson_folder)

    # Create the working directory if it doesn't exist
    if not os.path.exists(working_dir):
        os.makedirs(working_dir)
        print(f"Directory '{working_dir}' created.")
    else:
        print(f"Directory '{working_dir}' already exists.")

    # Create the lesson directory if it doesn't exist
    if not os.path.exists(lesson_dir):
        os.makedirs(lesson_dir)
        print(f"Directory '{lesson_dir}' created.")
    else:
        print(f"Directory '{lesson_dir}' already exists.")

    # Change the current working directory to the lesson directory
    os.chdir(lesson_dir)

    print(f"Current working directory: {os.getcwd()}")
else:
    print("Not connecting to Google Drive.")

In [ ]:
#@title Install Packages {display-mode:"form"}
INSTALL_PACKAGES = True #@param {type:"boolean"}

import os

# Check if packages have already been installed in this session to prevent re-installation
if INSTALL_PACKAGES and not os.environ.get('PYPSA_PACKAGES_INSTALLED'):
  !pip install git+https://github.com/pypsa/pypsa
  !pip install pypsa[excel]
  !pip install folium mapclassify cartopy gurobipy
  os.environ['PYPSA_PACKAGES_INSTALLED'] = 'true'
elif not INSTALL_PACKAGES:
  print("Skipping package installation.")
else:
  print("PyPSA packages are already installed for this session.")

# Modelling Exercise

#### Case Description

This case focussed on accessing the PyPSA network using the code based API. An example network is used from the PyPSA example library which demonstrates how to optimise meshed AC-DC networks in PyPSA. The example has a 3-node AC network coupled via AC-DC converters to a 3-node DC network. There is also a single point-to-point DC connection using the Link component.

For more information on this example, see the documentation.

[Meshed AC-DC Network Example](https://docs.pypsa.org/latest/examples/ac-dc-lopf/)

##### PyPSA Features Used

- Creating a network
- Accessing the components using the API
- Viewing static data
- View dynamic data
- Run the optimiser
- Inspect results
- Plot results


##### Non-Standard PyPSA Features

This notebook will only use standard features in PyPSA.

### Lesson Task

Access the static and dynamic data in each cell.

* Identify the data types
* Access specific columns and rows in the data.
* Plot data.


1. Create a network.

In [ ]:
import pypsa
pypsa.options.api.new_components_api = True

n = pypsa.examples.ac_dc_meshed()

2. Export the network to Excel and inspect the data.


To view the Excel file:
* Open a new tab
* Go to [drive.google.com](https://drive.google.com)
* Navigate to mydrive → ich-modelling-2026 → lesson-1

In [ ]:
n.export_to_excel('ac_dc_meshed.xlsx')

3. Inspect the snapshots

In [ ]:
n.snapshots

4. View all the components.

In [ ]:
n.components

In [ ]:
for i in n.components.keys():
    print(i)

In PyPSA, components are accessed using two naming conventions:

**Singular capitalized form** ('Bus', 'Generator', 'Load', etc.) is used when adding new components to the network using methods like `n.add()` or `n.components["Generator"].add()`.

**Plural lowercase form** (buses, generators, loads, etc.) is used to access component data. Using the new API, you can access:
- **Static data** by adding `.static` after the component name: `n.generators.static` shows all generator attributes
- **Time-varying data** by adding `.dynamic` after the component name: `n.generators.dynamic` returns a dictionary of DataFrames, where keys are the time-varying attributes

We will use the **plural lowercase form** to inspect the network.

5. View static data using `n.<component_name>.static`

* View the tabular data to explore the components.


6. Create visualizations to explore component attributes:

    **Generator Attributes:**
    ```python
    # Pie chart of generator capacities
    n.generators.static.p_nom.plot.pie(
        title="Generator Capacities",
        figsize=(8, 6),
        ylabel='',
        autopct=lambda pct: f'{pct*n.generators.static.p_nom.sum()/100:.0f} MW'
        )

    # Bar chart of generator efficiency
    n.generators.static.efficiency.plot.bar(title="Generator Efficiency", figsize=(10, 5))

    # Horizontal bar chart of generator capital costs
    n.generators.static.capital_cost.plot.barh(title="Generator Capital Costs", figsize=(10, 6))
    ```

    **Line Attributes:**
    ```python
    # Bar chart of line resistance
    n.lines.static.r.plot.bar(title="Line Resistance", figsize=(10, 5))

    # Bar chart of line reactance
    n.lines.static.x.plot.bar(title="Line Reactance", figsize=(10, 5))

    # Bar chart of line capital costs
    n.lines.static.capital_cost.plot.bar(title="Line Capital Costs", figsize=(10, 5))
    ```

    **Link Attributes:**
    ```python
    # Horizontal bar chart of link capital costs
    n.links.static.capital_cost.plot.barh(title="Link Capital Costs", figsize=(10, 6))
    ```

7. Inspect available time-varying keys using `n.<component_name>.dynamic.keys()`
* Since `n.<component_name>.dynamic` is a dictionary, you can use all standard dictionary operations.
* Inspect carriers, buses, generators, loads, lines, links storage_units, global_constraints.


8. Inspect specific time series data using tables:

* `n.loads.dynamic.p_set` — load power set points over time.
* `n.generators.dynamic.p_max_pu` — generator maximum power availability over time.


9. Creating timeseries plots

**Stacked Area Plot**
A stacked area plot shows how multiple components contribute to a total over time. Each component is a different color, and they stack on top of each other.

Use case: See the total load and which loads are largest at each time step.

```python
network.loads.dynamic.p_set.plot.area(
    stacked=True,
    figsize=(14, 6),
    title="Generator Power Output Over Time (Stacked)",
    ylabel="Power (MW)",
    xlabel="Time"
)
plt.tight_layout()
plt.show()
```

**Normal Line Plot**

A normal line plot shows each component as a separate line. Lines can overlap, making it harder to see totals but easier to track individual components.

Use case: Compare individual maximum generation profiles side-by-side without the visual "stacking" effect.

```python
network.generators.dynamic.p_max_pu.plot(
    figsize=(14, 6),
    title="Link Power Flow - Bus0 to Bus1",
    ylabel="Power (MW)",
    xlabel="Time"
)
plt.tight_layout()
plt.show()
```

10. PyPSA also has a plotting functionality which allows you to visualise the results on a map as long as the locations of each bus is given.

_Geospacial mapping is not covered in this course, but would be a useful skill to develop._

In [ ]:
line_color = n.lines.static.bus0.map(n.buses.static.carrier).map(
    lambda ct: "r" if ct == "DC" else "b"
)

n.plot.explore(
    line_color=line_color,
    link_color="c",
    jitter=0.4,
)

11. Solve the model

In [ ]:
n.optimize()

12. Inspect the results using Excel.

In [ ]:
n.export_to_excel('ac_dc_meshed_solved.xlsx')

13. Programatically view results using plots.


Using a stacked area plot:
- `network.generators.dynamic.p`

Using a normal line plot:
- `network.links.dynamic.p0`
- `network.links.dynamic.p1`
- `network.lines.dynamic.p0`
- `network.lines.dynamic.p1`



---